# 04 — DeFi Merge: Liquidations → Econometric Panel

**Purpose:** Merge the hourly DeFi liquidation data into the pre-DeFi econometric panel built in notebook 03. This produces the final analysis-ready dataset.

**Inputs:**
- `data/econ/econ_core_predefi_1h.parquet` (from notebook 03)
- `data/raw/defi/defi_liquidations_1h_clean.csv` (from Dune extraction)

**Output:**
- `data/econ/econ_core_full_1h.parquet` — the complete panel with DeFi variables
- `data/normalized/defi/liquidations_1h.parquet` — normalized DeFi series (reusable)
- `data/analysis/reports/defi_merge_qa.json`

**DeFi variable definitions:**
- `liq_usd_total` = total USD debt repaid via forced liquidations (primary)
- `liq_usd_collateral` = total USD collateral seized (robustness)
- `log_liq` = ln(1 + liq_usd_total) — log-transformed for regressions
- `n_liquidations` = event count (informational, not used in main spec)
- `liq_regime` = 1 if liq_usd_total > 95th percentile of non-zero hours

**Scope:** All chains, ETH-collateralized positions only. Excludes UwuLend post-hack (2024-06-10+) and Euler V1 hack window (2023-03-13 to 2023-04-12). Dust filter: events < $10 excluded.

In [1]:
# ── Setup ──
import sys; sys.path.insert(0, "..")
from config import CFG, REPORTS_DIR, ECON_DIR
CFG.ensure_dirs()

import pandas as pd
import numpy as np
import json

print(f"Project root: {CFG.ROOT}")

Project root: /Users/mirellaengerran/Desktop/Documents/Research_paper_leverage


In [3]:
# ── 1. Load pre-DeFi panel ──

panel = pd.read_parquet(CFG.FILES.econ_core_predefi, engine="pyarrow")
panel["date"] = pd.to_datetime(panel["date"], utc=True)
print(f"Pre-DeFi panel: {len(panel):,} rows × {panel.shape[1]} cols")

# Drop placeholder DeFi columns from notebook 03
drop_cols = [c for c in ["liq_usd_total", "log_liq"] if c in panel.columns]
panel = panel.drop(columns=drop_cols)
print(f"Dropped placeholders: {drop_cols}")

Pre-DeFi panel: 41,328 rows × 21 cols
Dropped placeholders: ['liq_usd_total', 'log_liq']


In [14]:
# ── 2. Load & normalize DeFi liquidation data ──

DEFI_RAW_PATH = CFG.ROOT / "data" / "raw" / "defi" / "defi_liquidations_1h_clean.csv"

defi = pd.read_csv(DEFI_RAW_PATH)
defi["date"] = pd.to_datetime(defi["date"], utc=True).dt.floor("h")

# Rename to standard column names
defi = defi.rename(columns={
    "total_debt_repaid_usd": "liq_usd_total",
    "total_collateral_seized_usd": "liq_usd_collateral",
})

print(f"DeFi raw: {len(defi):,} rows (hours with ≥1 liquidation)")
print(f"Date range: [{defi['date'].min()}, {defi['date'].max()}]")
print(f"Nulls: liq_usd_total={defi['liq_usd_total'].isna().sum()}, "
      f"liq_usd_collateral={defi['liq_usd_collateral'].isna().sum()}")

# Basic sanity
assert defi["date"].duplicated().sum() == 0, "Duplicate hours in DeFi data!"
assert (defi["liq_usd_total"].dropna() >= 0).all(), "Negative debt values!"
print("✅ DeFi data passes sanity checks")

DeFi raw: 10,976 rows (hours with ≥1 liquidation)
Date range: [2021-03-15 05:00:00+00:00, 2025-11-30 23:00:00+00:00]
Nulls: liq_usd_total=0, liq_usd_collateral=0
✅ DeFi data passes sanity checks


In [16]:
# ── 3. Merge onto panel (LEFT JOIN → hours without liq get NaN → fill 0) ──

panel = panel.merge(
    defi[["date", "liq_usd_total", "liq_usd_collateral", "n_liquidations"]],
    on="date",
    how="left"
)

# Fill hours with no liquidations → 0 (not NaN — zero is the true value)
for col in ["liq_usd_total", "liq_usd_collateral", "n_liquidations"]:
    n_missing = panel[col].isna().sum()
    panel[col] = panel[col].fillna(0)
    print(f"  {col}: {n_missing:,} hours filled with 0 ({100*n_missing/len(panel):.1f}%)")

print(f"\nPanel after merge: {len(panel):,} rows × {panel.shape[1]} cols")

  liq_usd_total: 30,352 hours filled with 0 (73.4%)
  liq_usd_collateral: 30,352 hours filled with 0 (73.4%)
  n_liquidations: 30,352 hours filled with 0 (73.4%)

Panel after merge: 41,328 rows × 22 cols


In [18]:
# ── 4. Construct DeFi-derived variables ──

# 4a. Log-transformation (primary regressor)
panel["log_liq"] = np.log1p(panel["liq_usd_total"])

# 4b. Lagged shock (used in local projections — reduces reverse causality)
panel["log_liq_lag1"] = panel["log_liq"].shift(1)

# 4c. Stress regime: top 5% of NON-ZERO liquidation hours
nonzero_liq = panel.loc[panel["liq_usd_total"] > 0, "liq_usd_total"]
stress_threshold = nonzero_liq.quantile(CFG.ECON.stress_pctile / 100)
panel["liq_stress"] = (panel["liq_usd_total"] > stress_threshold).astype(int)

# 4d. Interaction: lagged shock × OI regime (key test of leverage cycle)
panel["shock_x_oi"] = panel["log_liq_lag1"] * panel["oi_high"]

print(f"Log-liq: mean={panel['log_liq'].mean():.3f}, max={panel['log_liq'].max():.1f}")
print(f"Stress threshold (P{CFG.ECON.stress_pctile} of non-zero): ${stress_threshold:,.0f}")
print(f"Hours in stress regime: {panel['liq_stress'].sum():,} ({100*panel['liq_stress'].mean():.1f}%)")
print(f"Hours with zero liquidations: {(panel['liq_usd_total'] == 0).sum():,}")

Log-liq: mean=1.724, max=19.1
Stress threshold (P95 of non-zero): $261,577
Hours in stress regime: 549 (1.3%)
Hours with zero liquidations: 30,352


In [20]:
# ── 5. QA: cross-check with known events ──

print("═══ Known Event Cross-Check ═══\n")
events = {
    "May 19 2021 crash":           ("2021-05-19", "2021-05-20"),
    "Dec 4 2021 flash crash":      ("2021-12-03", "2021-12-05"),
    "Jun 2022 Terra/Luna":         ("2022-06-12", "2022-06-15"),
    "Nov 2022 FTX":                ("2022-11-08", "2022-11-11"),
    "Aug 5 2024 yen carry unwind": ("2024-08-05", "2024-08-06"),
    "Feb 3 2025 DeepSeek":         ("2025-02-02", "2025-02-04"),
}

for name, (start, end) in events.items():
    mask = (panel["date"] >= start) & (panel["date"] < end)
    sub = panel[mask]
    total_liq = sub["liq_usd_total"].sum()
    peak_liq = sub["liq_usd_total"].max()
    mean_ret = sub["ret_eth_perp"].mean()
    print(f"{name:35s}: liq=${total_liq/1e6:>7.1f}M  peak=${peak_liq/1e6:>7.1f}M/h  "
          f"avg_ret={mean_ret:>+.3f}%/h")

═══ Known Event Cross-Check ═══

May 19 2021 crash                  : liq=$   92.8M  peak=$   23.8M/h  avg_ret=-1.349%/h
Dec 4 2021 flash crash             : liq=$   30.9M  peak=$   26.5M/h  avg_ret=-0.192%/h
Jun 2022 Terra/Luna                : liq=$   85.1M  peak=$   31.2M/h  avg_ret=-0.330%/h
Nov 2022 FTX                       : liq=$   14.1M  peak=$    7.7M/h  avg_ret=-0.261%/h
Aug 5 2024 yen carry unwind        : liq=$  307.2M  peak=$  187.6M/h  avg_ret=-0.440%/h
Feb 3 2025 DeepSeek                : liq=$  246.5M  peak=$  130.4M/h  avg_ret=-0.165%/h


In [22]:
# ── 6. Final column inventory ──

print("═══ Final Panel Columns ═══\n")
col_groups = {
    "Identifiers": ["date"],
    "Prices & returns": ["close_perp", "ret_eth_perp", "close_btc_spot", "ret_btc_spot",
                          "close_eth_spot", "ret_eth_spot"],
    "Leverage proxies": ["oi", "d_oi", "oi_zscore", "oi_high", "oi_vol_ratio",
                          "funding_rate", "basis_bps"],
    "Volatility": ["vol_eth_7d", "vol_btc_7d", "ret_eth_std", "ret_btc_std"],
    "DeFi liquidations": ["liq_usd_total", "liq_usd_collateral", "n_liquidations",
                           "log_liq", "log_liq_lag1", "liq_stress", "shock_x_oi"],
    "Volume": ["volume_perp"],
}

for group, cols in col_groups.items():
    present = [c for c in cols if c in panel.columns]
    missing = [c for c in cols if c not in panel.columns]
    print(f"  {group}: {len(present)} present" + (f", {len(missing)} missing: {missing}" if missing else ""))

print(f"\nTotal: {panel.shape[1]} columns")

═══ Final Panel Columns ═══

  Identifiers: 1 present
  Prices & returns: 6 present
  Leverage proxies: 7 present
  Volatility: 4 present
  DeFi liquidations: 7 present
  Volume: 1 present

Total: 26 columns


In [24]:
# ── 7. Save ──

# Full econometric panel
panel.to_parquet(CFG.FILES.econ_core_full, index=False, engine="pyarrow")
print(f"Saved: {CFG.FILES.econ_core_full}")

# Normalized DeFi series (reusable independently)
defi_norm = panel[["date", "liq_usd_total", "liq_usd_collateral", "n_liquidations"]].copy()
defi_norm.to_parquet(CFG.FILES.defi_liq, index=False, engine="pyarrow")
print(f"Saved: {CFG.FILES.defi_liq}")

# QA report
qa = {
    "panel_rows": len(panel),
    "panel_cols": panel.shape[1],
    "defi_raw_rows": len(defi),
    "hours_with_liquidations": int((panel["liq_usd_total"] > 0).sum()),
    "hours_zero_liquidations": int((panel["liq_usd_total"] == 0).sum()),
    "stress_threshold_usd": float(stress_threshold),
    "hours_in_stress": int(panel["liq_stress"].sum()),
    "total_liq_usd": float(panel["liq_usd_total"].sum()),
    "max_hourly_liq_usd": float(panel["liq_usd_total"].max()),
    "nulls_after_merge": {c: int(panel[c].isna().sum()) for c in
                          ["liq_usd_total", "log_liq", "ret_eth_perp", "oi"]},
    "status": "PASS",
}

qa_path = REPORTS_DIR / "defi_merge_qa.json"
with open(qa_path, "w") as f:
    json.dump(qa, f, indent=2)
print(f"Saved: {qa_path}")

print(f"\n✅ Notebook 04 complete — panel ready for estimation")
print(f"   Next: re-run notebook 07 (quantile LP) with real DeFi data")

Saved: /Users/mirellaengerran/Desktop/Documents/Research_paper_leverage/data/econ/econ_core_full_1h.parquet
Saved: /Users/mirellaengerran/Desktop/Documents/Research_paper_leverage/data/normalized/defi/liquidations_1h.parquet
Saved: /Users/mirellaengerran/Desktop/Documents/Research_paper_leverage/data/analysis/reports/defi_merge_qa.json

✅ Notebook 04 complete — panel ready for estimation
   Next: re-run notebook 07 (quantile LP) with real DeFi data
